In [0]:
%pip install phonenumbers pycountry rapidfuzz

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from utils.silver_functions import *

# Lecture Bronze
# Les id sont changés dans la phase de silver_suppliers
incidents = spark.table("workspace.final_project.bronze_incidents")
display(incidents)
incidents_quarantine = (
    incidents.select([col(c).cast("string").alias(c) for c in incidents.columns])
    .limit(0)
    .withColumn("quarantine_reason", lit(None).cast("string"))
    .withColumn("quarantine_timestamp", lit(None).cast("string"))
)
display(incidents_quarantine)
# Il faudra prendre les silvers quand ils seront prets.
suppliers = spark.table("workspace.final_project.silver_suppliers")
orders = spark.table("workspace.final_project.silver_orders")


In [0]:
# Controle de id

incidents, duplicate_incident_id = check_duplicate_id(incidents, "incident_id")
incidents_quarantine = incidents_quarantine.unionByName(cast_all_columns_to_string(duplicate_incident_id))

In [0]:
# order_id: normalisation int + quanrantaine des nulls (incidents doit être rattaché a un order)
# + vérification présence dans la table bronze_orders 

incidents, invalid_order_id = clean_integer_column(incidents, 
                                                   "order_id", 
                                                   to_delete=True, 
                                                   accept_float=False)

incidents_quarantine = incidents_quarantine.unionByName(cast_all_columns_to_string(invalid_order_id))
incidents, invalid_order_not_found = validate_foreign_key(incidents, 
                                                          orders, 
                                                          "order_id", 
                                                          "order_id")

incidents_quarantine = incidents_quarantine.unionByName(cast_all_columns_to_string(invalid_order_not_found))

In [0]:
# incident_type et severity: normalisation string
# incident_date: normalisation date
incidents = (
    incidents
    .withColumn("incident_type", normalize_string_udf("incident_type"))
    .withColumn("severity", normalize_string_udf(col("severity")))
    .withColumn("incident_date", normalize_date("incident_date"))
    .withColumn("description", trim(col("description")))
)

In [0]:
incidents.show()

In [0]:
incidents_quarantine.show()

In [0]:
incidents.write.mode("overwrite").saveAsTable(
    "workspace.final_project.silver_incidents"
)
incidents_quarantine.write.mode("overwrite").saveAsTable(
    "workspace.final_project.incidents_quarantine"
)